# DSPy Cybersecurity SOC Assistant

## What this project demonstrates

DSPy replaces hand-crafted prompts with **declarative programming**. You declare *what* you want an LM to do using typed Signatures, compose them into Modules, and let DSPy compile and optimize the prompts automatically.

This notebook walks through three production-style cybersecurity pipelines:

| Pipeline | Task | Key DSPy concept |
|---|---|---|
| 1. Threat Classifier | Classify network logs (DDoS, PortScan, etc.) | `Signature` + `ChainOfThought` |
| 2. CVE Analyst | Assess CVE impact on infrastructure | Module composition (3 steps) |
| 3. Alert Triage | Prioritize SOC alert queues | `Assert` constraints + optimization |

**All free. All local. No API keys.**
Powered by Ollama running on your machine.

In [ ]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'dspy-ai>=2.4', 'datasets', 'pandas', 'scikit-learn', 'requests', 'tqdm'])
print('Installed.')

In [ ]:
# ── Cell 2: Paths + imports ──────────────────────────────────────────────────
import sys, os, json
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import dspy
print(f'DSPy {dspy.__version__}')

In [ ]:
# ── Cell 3: Connect to Ollama ────────────────────────────────────────────────
# Requires: ollama serve && ollama pull llama3.2

from src.utils.utils import configure_lm
model = configure_lm()  # auto-selects best available model

---
## Part 1: DSPy Concepts — Signatures, Predict, ChainOfThought

Before running the full pipelines, let's understand the building blocks.

In [ ]:
# ── Cell 4: What is a Signature? ─────────────────────────────────────────────
#
# A Signature is NOT a prompt. It's a typed contract:
#   - InputField:  what data goes in
#   - OutputField: what data comes out
#   - docstring:   the task description
#
# DSPy compiles it into a prompt automatically.

from src.signatures.threat import ThreatClassifier, ThreatClassifierWithReasoning

print('=== ThreatClassifier Signature ===')
print(f'Task: {ThreatClassifier.__doc__.strip()[:100]}...')
print(f'Fields: {list(ThreatClassifier.model_fields.keys())}')

print('\n=== Predict vs ChainOfThought ===')
print('dspy.Predict(ThreatClassifier):')
print('  One LM call. Outputs: threat_type, confidence, key_indicators')
print('  No reasoning trace.')
print()
print('dspy.ChainOfThought(ThreatClassifierWithReasoning):')
print('  One LM call. Outputs: REASONING (auto-added), then threat_type, confidence, ...')
print('  Model thinks step-by-step before answering.')

In [ ]:
# ── Cell 5: Predict vs ChainOfThought — side-by-side ─────────────────────────

test_log = (
    'proto=TCP src=185.220.101.42 dst=10.0.0.8 dport=22 '
    'pkts=3200 bytes=98000 dur=45s flags=S service=SSH attempts=3200'
)

# Basic Predict — no reasoning
basic = dspy.Predict(ThreatClassifier)
pred_basic = basic(log_entry=test_log)

print('=== dspy.Predict (no CoT) ===')
print(f'threat_type:    {pred_basic.threat_type}')
print(f'confidence:     {pred_basic.confidence}')
print(f'key_indicators: {pred_basic.key_indicators[:100]}')
print(f'reasoning:      [not available — use ChainOfThought for this]')

print()

# ChainOfThought — adds reasoning automatically
cot = dspy.ChainOfThought(ThreatClassifierWithReasoning)
pred_cot = cot(log_entry=test_log)

print('=== dspy.ChainOfThought ===')
print(f'reasoning (auto-added):')
print(f'  {pred_cot.reasoning[:300]}')
print(f'\nthreat_type:    {pred_cot.threat_type}')
print(f'confidence:     {pred_cot.confidence}')
print(f'mitre_tactic:   {pred_cot.mitre_tactic}')
print(f'action:         {pred_cot.recommended_action}')

In [ ]:
# ── Cell 6: Inspect what DSPy compiled ───────────────────────────────────────
print('=== Compiled prompt DSPy sent to Ollama ===')
print('(Auto-generated from your Signature — you never wrote this)')
print('-' * 60)
dspy.inspect_history(n=1)

---
## Part 2: Pipeline 1 — Threat Classifier

Classifies real network log entries from CICIDS 2017 into threat categories.

In [ ]:
# ── Cell 7: Load CICIDS 2017 dataset ─────────────────────────────────────────
from data.loader import load_threat_dataset

# max_samples_per_class=30 for quick demo; increase for full training
threat_train, threat_test = load_threat_dataset(max_samples_per_class=30)

print(f'Train: {len(threat_train)} | Test: {len(threat_test)}')
print('\nSample examples:')
for ex in threat_train[:3]:
    print(f'  [{ex.threat_type:12s}] {ex.log_entry[:80]}')

In [ ]:
# ── Cell 8: Run Threat Classifier ────────────────────────────────────────────
from src.modules.all_modules import ThreatClassifierModule, BatchThreatClassifier

classifier = ThreatClassifierModule()

test_logs = [
    'proto=TCP src=185.220.101.42 dst=10.0.0.8 dport=22 pkts=3200 bytes=98000 dur=45s flags=S service=SSH attempts=3200',
    'proto=TCP src=45.33.32.156 dst=10.0.0.5 dport=8080 pkts=1 bytes=40 dur=0.001s flags=S',
    'proto=TCP src=10.0.0.5 dst=216.58.215.14 dport=443 pkts=18 bytes=8400 dur=2.1s flags=ACK tls=True',
    'proto=UDP src=203.0.113.4 dst=10.0.0.1 dport=80 pkts=50000 bytes=3200000 dur=0.3s flood=True',
    'proto=TCP src=10.0.0.12 dst=185.220.4.5 dport=6667 pkts=240 bytes=18000 dur=120s c2=True beacon=True',
]

print('=== Threat Classification Results ===')
for log in test_logs:
    report = classifier(log_entry=log)
    bar = '█' * int(report.confidence * 8) + '░' * (8 - int(report.confidence * 8))
    print(f'  [{report.threat_type:12s}] [{bar}] {report.confidence:.2f}')
    print(f'    Log:     {log[:70]}')
    print(f'    Action:  {report.recommended_action}')
    print()

In [ ]:
# ── Cell 9: Evaluate Threat Classifier on test set ───────────────────────────
from src.utils.utils import evaluate_pipeline, threat_accuracy_metric

metrics = evaluate_pipeline(
    module=classifier,
    test_examples=threat_test,
    metric_fn=threat_accuracy_metric,
    pipeline_name='Threat Classifier',
    n_samples=12,   # increase for full eval
)
print('\nMetrics:', metrics)

---
## Part 3: Pipeline 2 — CVE Analyst

Analyzes CVEs with real NIST NVD data — severity classification, infrastructure impact assessment, and remediation planning in a 3-step composed module.

In [ ]:
# ── Cell 10: Load CVE dataset ────────────────────────────────────────────────
from data.loader import load_cve_dataset

cve_train, cve_test = load_cve_dataset(max_cves=40)

print(f'CVE Train: {len(cve_train)} | Test: {len(cve_test)}')
print('\nSample CVEs:')
for ex in cve_test[:3]:
    print(f'  {ex.cve_id} (CVSS={ex.cvss_score}) → {ex.severity}')
    print(f'    {ex.cve_description[:100]}...')

In [ ]:
# ── Cell 11: Run CVE Analyst ─────────────────────────────────────────────────
from src.modules.all_modules import CVEAnalystModule

analyst = CVEAnalystModule()

# Infrastructure context — describe YOUR environment
infra = (
    '500-person organization. Active Directory (Windows Server 2019). '
    'Apache 2.4 web servers on Ubuntu 20.04. Java 11 applications. '
    'Cisco ASA 9.x firewalls. Public-facing Confluence and GitLab instances. '
    'No WAF. Palo Alto NGFW at perimeter.'
)

# Analyze Log4Shell (CVE-2021-44228)
log4shell = cve_test[0]

print(f'Analyzing: {log4shell.cve_id}')
print(f'Description: {log4shell.cve_description[:120]}...')
print('-' * 60)

report = analyst(
    cve_id=log4shell.cve_id,
    cve_description=log4shell.cve_description,
    cvss_score=log4shell.cvss_score,
    infrastructure_context=infra,
)

print(f'Severity:               {report.severity}')
print(f'CVSS:                   {report.cvss_score}')
print(f'Affected component:     {report.affected_component}')
print(f'Attack vector:          {report.attack_vector}')
print(f'Requires auth:          {report.requires_auth}')
print(f'Impact:                 {report.impact_summary}')
print()
print(f'Likely affected:        {report.likely_affected}')
print(f'Affected assets:        {report.affected_assets}')
print(f'Blast radius:           {report.blast_radius}')
print(f'Exploitation:           {report.exploitation_likelihood}')
print(f'Patch urgency:          {report.patch_urgency}')
print()
print(f'Immediate actions:')
print(f'  {report.immediate_actions}')
print()
print(f'Short-term fix:  {report.short_term_fix}')
print(f'Long-term fix:   {report.long_term_fix}')
print(f'Detection query: {report.detection_query}')
print(f'Effort:          {report.effort_estimate}')

---
## Part 4: Pipeline 3 — Alert Triage

Takes a mixed queue of SOC alerts and produces a prioritized list with analyst notes.

In [ ]:
# ── Cell 12: Run Alert Triage ────────────────────────────────────────────────
from src.modules.all_modules import AlertTriageModule
from data.loader import load_alert_dataset

alert_train, alert_test = load_alert_dataset(n_train=20, n_test=5)
triage = AlertTriageModule()

# Run on first test example
ex = alert_test[0]
print('=== Alert Queue (input) ===')
queue = json.loads(ex.alert_queue)
for a in queue:
    print(f'  [{a["severity"]:13s}] {a["title"]} ({a["source"]})')

print('\n=== Running Triage... ===')
result = triage(alert_queue=ex.alert_queue)

print('\n=== Triage Results ===')
print(f'Priority order:   {result.priority_order}')
print(f'Top threat:       {result.top_threat_summary}')
print(f'Escalate now:     {result.escalate_immediately}')
print(f'Correlated:       {result.correlated_alerts}')
print()
print(f'Top alert verdict:   {result.top_alert_verdict}')
print(f'Attack stage:        {result.attack_stage}')
print(f'IOCs:                {result.iocs}')
print(f'\nContainment steps:')
print(f'  {result.containment_steps}')
print(f'\nTicket summary:')
print(f'  {result.ticket_summary}')

---
## Part 5: DSPy Optimization — BootstrapFewShot

DSPy optimizers automatically improve your pipeline by selecting the best few-shot examples from your training data. No manual prompt tuning needed.

In [ ]:
# ── Cell 13: Optimize Threat Classifier with BootstrapFewShot ────────────────
#
# BootstrapFewShot:
#   1. Runs the unoptimized program on training examples
#   2. Identifies examples where the program succeeded
#   3. Uses those as few-shot demonstrations in the optimized prompt
#
# This is the DSPy equivalent of writing good few-shot examples by hand —
# but the optimizer selects them automatically using your metric function.

from dspy.teleprompt import BootstrapFewShot
from src.utils.utils import threat_accuracy_metric

print('=== BootstrapFewShot Optimization ===')
print('This automatically selects the best few-shot examples for your prompts.')
print('Using 10 training examples (increase for better results)...')

# Use a small subset for demo speed
mini_train = threat_train[:10]

optimizer = BootstrapFewShot(
    metric=threat_accuracy_metric,
    max_bootstrapped_demos=2,   # few-shot examples per predictor
    max_labeled_demos=2,
)

# Compile = optimize the program
optimized_classifier = optimizer.compile(
    student=ThreatClassifierModule(),
    trainset=mini_train,
)

print('\nOptimization complete!')
print('The optimized program now has auto-selected few-shot examples.')
print('Inspect the compiled prompt:')
dspy.inspect_history(n=1)

In [ ]:
# ── Cell 14: Compare baseline vs optimized ───────────────────────────────────
from src.utils.utils import evaluate_pipeline, threat_accuracy_metric

eval_set = threat_test[:8]

print('Baseline (no few-shot):')
baseline_metrics = evaluate_pipeline(
    module=ThreatClassifierModule(),
    test_examples=eval_set,
    metric_fn=threat_accuracy_metric,
    pipeline_name='Baseline',
    verbose=False,
)

print('\nOptimized (BootstrapFewShot):')
optimized_metrics = evaluate_pipeline(
    module=optimized_classifier,
    test_examples=eval_set,
    metric_fn=threat_accuracy_metric,
    pipeline_name='Optimized',
    verbose=False,
)

print('\n=== Comparison ===')
print(f'  Baseline:  {baseline_metrics["avg_score"]:.3f} ({baseline_metrics["avg_score"]*100:.1f}%)')
print(f'  Optimized: {optimized_metrics["avg_score"]:.3f} ({optimized_metrics["avg_score"]*100:.1f}%)')
delta = optimized_metrics['avg_score'] - baseline_metrics['avg_score']
print(f'  Delta:     {delta:+.3f} ({delta*100:+.1f}%)')

In [ ]:
# ── Cell 15: Save the optimized program ──────────────────────────────────────
save_path = str(PROJECT_ROOT / 'outputs' / 'optimized' / 'threat_classifier.json')
optimized_classifier.save(save_path)
print(f'Saved optimized program to: {save_path}')

# Load it back
loaded = ThreatClassifierModule()
loaded.load(save_path)
print('Loaded successfully. Ready for production use.')

---
## Summary: DSPy concepts covered

| Concept | File | What it does |
|---|---|---|
| `dspy.Signature` | `src/signatures/` | Declares LM contract: inputs → outputs |
| `dspy.InputField` | All signatures | Annotated input variable |
| `dspy.OutputField` | All signatures | Annotated output the LM must produce |
| `dspy.Predict` | `CVEAnalystModule` step 1/3 | Single LM call, no reasoning |
| `dspy.ChainOfThought` | `ThreatClassifierModule`, `CVEAnalystModule` step 2, `AlertTriageModule` | Auto-adds reasoning field |
| `dspy.Module` | All `*Module` classes | Composable pipeline unit |
| `dspy.Assert` | `ThreatClassifierModule`, `AlertTriageModule` | Runtime constraint on outputs |
| `dspy.Example` | `data/loader.py` | Labelled data point for eval/optimization |
| `BootstrapFewShot` | Cell 13 | Auto-selects few-shot demos from training data |
| `.save()` / `.load()` | Cell 15 | Serialize optimized program to disk |

### What to explore next

- **`MIPROv2`** — stronger optimizer that also rewrites the instruction text, not just few-shot examples
- **`dspy.TypedChainOfThought`** — enforce Pydantic types on outputs for fully structured JSON
- **`dspy.Suggest`** — soft constraints (doesn't retry on failure, unlike `Assert`)
- **Notebook 04** (`04_alert_triage.ipynb`) — shows `MIPROv2` optimization on the alert triage pipeline

### GitHub push checklist
```bash
git init
git add .
git commit -m 'feat: DSPy cybersecurity SOC assistant — three pipelines'
git branch -M main
git remote add origin https://github.com/YOUR_USERNAME/dspy-cybersec-soc
git push -u origin main
```